# Training Paper Checkpoints

This notebook is a compact, standalone record of the training script used to produce the paper checkpoints for the Voronoi and semi-supervised experiments. It trains attention-only in-context predictors in two variants:

- **Unconstrained:** the attention layer acts directly on the token representation.
- **Symmetrized:** each attention update is computed after random block-wise coordinate permutations and then mapped back to the original coordinates.

The code below is intentionally self-contained so that the task distributions, architecture, optimization settings, and checkpoint layout can be inspected without following project-internal training scripts.

## Imports

Only standard scientific Python packages, PyTorch, `tqdm`, and `yaml` are needed. Independent runs can be distributed across visible CUDA devices.

In [ ]:
import math
import time
import uuid
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm
import yaml

## Experiment Configuration

The configuration cell specifies the exact run grid: Voronoi and semi-supervised tasks, each trained with and without the symmetrized attention mechanism. Each configuration is repeated for two seeds. Checkpoints are written to `./models/{task}/{variant}/{run_id}`.

In [ ]:
RUN_NAME = "mytrain"
OUT_ROOT = "./models"
TEST_RUN = False
N_RUNS = 2
SEED = 4
DEVICE = "cuda"
PARALLEL_GPUS = True

TASK_MODE = ["voronoicell", "semi_supervised"]
PERMUTATIONS = [True, False]

N_CLASSES = 3
DIMS = 7
POINTS = 65

SEMI_ETA = 0.5
SEMI_N_LABELED = 8
SEMI_REPLACE_UNLABELED_WITH_NOISE = False
SEMI_TRAIN_POINT_CHOICES = [8, 16, 32, 64, 128]

N_LAYER = 5
N_HEAD = 1
ACT_ATTN = "softmax"
ATTN_MASK = "drop_last"

BATCH_SIZE = 8192
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BETA1 = 0.9
BETA2 = 0.999
TRAIN_STEPS = 20001
REDRAW_EVERY_ITERS = 1
SAVE_EVERY_STEPS = 1000
DTYPE = torch.float32

## Model Definition

The model is an attention-only residual transformer. The symmetrized variant applies random permutations separately to the feature block and label block before each attention update, then undoes those permutations before adding the update back to the residual stream.

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class MultiheadAttention(nn.Module):
    def __init__(self, dim_embed, n_heads):
        super().__init__()
        self.n_heads = int(n_heads)
        self.dim_head = int(dim_embed)
        self.projQ = nn.Linear(dim_embed, n_heads * dim_embed, bias=False, dtype=DTYPE)
        self.projK = nn.Linear(dim_embed, n_heads * dim_embed, bias=False, dtype=DTYPE)
        self.projV = nn.Linear(dim_embed, n_heads * dim_embed, bias=False, dtype=DTYPE)
        self.projFinal = nn.Linear(n_heads * dim_embed, dim_embed, bias=False, dtype=DTYPE)
        self._init_weights()

    def _init_weights(self):
        for module in [self.projQ, self.projK, self.projV]:
            k = 1.0 / module.weight.shape[1]
            torch.nn.init.uniform_(module.weight, -k, k)
        torch.nn.init.uniform_(self.projFinal.weight)

    def forward(self, query, key, value):
        bsize, n_tokens = query.shape[:2]
        query = self.projQ(query).view(bsize, n_tokens, self.n_heads, self.dim_head).transpose(1, 2)
        key = self.projK(key).view(bsize, n_tokens, self.n_heads, self.dim_head).transpose(1, 2)
        value = self.projV(value).view(bsize, n_tokens, self.n_heads, self.dim_head).transpose(1, 2)

        mask = torch.ones((n_tokens, n_tokens), dtype=torch.bool, device=query.device)
        mask[:, -1] = False
        attn = query @ key.transpose(-2, -1) / math.sqrt(self.dim_head)
        attn = attn.masked_fill(~mask[None, None, :, :], float("-inf"))
        attn = nn.functional.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0, posinf=0.0, neginf=0.0)

        out = attn @ value
        out = out.transpose(1, 2).contiguous().view(bsize, n_tokens, self.n_heads * self.dim_head)
        return self.projFinal(out)


class Transformer(nn.Module):
    def __init__(self, n_heads, n_layers, dim_embed, permutations=False, permutations_kfix=None):
        super().__init__()
        self.permutations = bool(permutations)
        self.permutations_kfix = int(permutations_kfix)
        self.layers_attn = nn.ModuleList([MultiheadAttention(dim_embed, n_heads) for _ in range(n_layers)])

    @staticmethod
    def permute_blocks(z, k: int):
        bsize, n, d = z.shape
        m = d - int(k)
        z_perm = torch.empty_like(z)

        perms1 = torch.argsort(torch.rand(bsize, m, device=z.device), dim=1)
        idx1 = perms1[:, None, :].expand(-1, n, -1)
        z_perm[:, :, :m] = torch.gather(z[:, :, :m], 2, idx1)

        perms2 = torch.argsort(torch.rand(bsize, k, device=z.device), dim=1)
        idx2 = perms2[:, None, :].expand(-1, n, -1)
        z_perm[:, :, m:] = torch.gather(z[:, :, m:], 2, idx2)
        return z_perm, perms1, perms2

    @staticmethod
    def undo_permute_blocks(z_perm, perms1, perms2, k: int):
        bsize, n, d = z_perm.shape
        m = d - int(k)
        z_rec = torch.empty_like(z_perm)

        inv1 = torch.empty_like(perms1)
        inv1.scatter_(1, perms1, torch.arange(m, device=z_perm.device).expand(bsize, -1))
        z_rec[:, :, :m] = torch.gather(z_perm[:, :, :m], 2, inv1[:, None, :].expand(-1, n, -1))

        inv2 = torch.empty_like(perms2)
        inv2.scatter_(1, perms2, torch.arange(k, device=z_perm.device).expand(bsize, -1))
        z_rec[:, :, m:] = torch.gather(z_perm[:, :, m:], 2, inv2[:, None, :].expand(-1, n, -1))
        return z_rec

    def forward(self, x):
        for attn_layer in self.layers_attn:
            if self.permutations:
                xp, p1, p2 = self.permute_blocks(x, k=self.permutations_kfix)
                update = self.undo_permute_blocks(attn_layer(xp, xp, xp), p1, p2, k=self.permutations_kfix)
            else:
                update = attn_layer(x, x, x)
            x = x + update
        return x


class ICPredictor(nn.Module):
    def __init__(self, permutations=False):
        super().__init__()
        self.transformer = Transformer(
            n_heads=N_HEAD,
            n_layers=N_LAYER,
            dim_embed=DIMS + N_CLASSES,
            permutations=permutations,
            permutations_kfix=N_CLASSES,
        )

    def forward(self, x):
        return self.transformer(x)


def build_model(permutations=False):
    return ICPredictor(permutations=bool(permutations))

## Task Distributions

The semi-supervised task is a linear classification problem with only eight labeled context points and a label-dependent feature shift. During training, the total context size is sampled from a fixed set. The Voronoi task labels each point by its nearest randomly sampled class site; all context labels are visible and the query label is hidden.

In [ ]:
class SemiSupervisedLinearClassification:
    def __init__(self, bsize, n_points, n_dims, device):
        self.bsize = int(bsize)
        self.n_points = int(n_points)
        self.n_dims = int(n_dims)
        self.device = device
        self.W = None

    def resample(self):
        W = torch.randn(self.bsize, self.n_dims, N_CLASSES, device=self.device, dtype=DTYPE)
        self.W = W / (W.norm(dim=1, keepdim=True) + 1e-8)

    def evaluate(self, n_points=None):
        n = int(self.n_points if n_points is None else n_points)
        x = torch.randn(self.bsize, n, self.n_dims, device=self.device, dtype=DTYPE)
        W = self.W.to(device=x.device, dtype=x.dtype)
        class_idx = (x @ W).argmax(dim=-1)

        W_T = W.transpose(1, 2)
        batch_idx = torch.arange(self.bsize, device=x.device)[:, None]
        x = x + SEMI_ETA * W_T[batch_idx, class_idx]

        y_true = torch.zeros(self.bsize, n, N_CLASSES, device=x.device, dtype=x.dtype)
        y_true.scatter_(2, class_idx.unsqueeze(-1), 1.0)
        y = y_true.clone()

        keep = torch.zeros(self.bsize, n, device=x.device, dtype=torch.bool)
        keep[:, -1] = True
        scores = torch.rand(self.bsize, n, device=x.device)
        scores[:, -1] = -1.0
        keep.scatter_(1, scores.argsort(dim=-1, descending=True)[:, :SEMI_N_LABELED], True)
        hide = ~keep
        hide[:, -1] = False
        y = y.masked_fill(hide.unsqueeze(-1), 0.0)

        if SEMI_REPLACE_UNLABELED_WITH_NOISE:
            unlabeled = y.abs().sum(dim=-1) == 0
            x = torch.where(unlabeled.unsqueeze(-1), torch.randn_like(x), x)

        y[:, -1, :] = 0.0
        return torch.cat([x, y], dim=-1), y_true[:, -1, :]


class VoronoiClassification:
    def __init__(self, bsize, n_points, n_dims, device):
        self.bsize = int(bsize)
        self.n_points = int(n_points)
        self.n_dims = int(n_dims)
        self.device = device
        self.C = None

    def resample(self):
        self.C = torch.randn(self.bsize, self.n_dims, N_CLASSES, device=self.device, dtype=DTYPE)

    def evaluate(self, n_points=None):
        n = int(self.n_points if n_points is None else n_points)
        x = torch.randn(self.bsize, n, self.n_dims, device=self.device, dtype=DTYPE)
        C = self.C.to(device=x.device, dtype=x.dtype)
        C_T = C.transpose(1, 2)

        x2 = (x * x).sum(dim=-1, keepdim=True)
        c2 = (C_T * C_T).sum(dim=-1).unsqueeze(1)
        d2 = x2 + c2 - 2.0 * (x @ C)
        class_idx = d2.argmin(dim=-1)

        y_true = torch.zeros(self.bsize, n, N_CLASSES, device=x.device, dtype=x.dtype)
        y_true.scatter_(2, class_idx.unsqueeze(-1), 1.0)
        z = torch.cat([x, y_true], dim=-1)
        z[:, -1, -N_CLASSES:] = 0.0
        return z, y_true[:, -1, :]


def build_task(mode, bsize, n_points, n_dims, device):
    if mode == "semi_supervised":
        return SemiSupervisedLinearClassification(bsize, n_points, n_dims, device=device)
    if mode == "voronoicell":
        return VoronoiClassification(bsize, n_points, n_dims, device=device)
    raise ValueError(f"Unknown task mode: {mode}")


def sample_total_tokens(mode):
    if mode != "semi_supervised":
        return int(POINTS)
    total_choices = [int(v) + 1 for v in SEMI_TRAIN_POINT_CHOICES]
    valid = [v for v in total_choices if SEMI_N_LABELED + 1 <= v <= POINTS]
    return int(valid[torch.randint(0, len(valid), (1,), dtype=torch.long).item()])

## Training Utilities

Each run redraws a fresh task instance before every optimization step, trains with cross-entropy on the query token, and periodically overwrites `state.pt` in the run directory. The saved YAML config mirrors the scalar constants above so downstream analysis can reconstruct the training distribution.

In [ ]:
def train_step(model, z, y, optimizer):
    optimizer.zero_grad(set_to_none=True)
    out = model(z)
    logits = out[:, -1, -y.shape[-1]:]
    loss = torch.nn.functional.cross_entropy(logits, y.argmax(dim=-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return float(loss.detach().item())


def resolve_devices():
    if not str(DEVICE).startswith("cuda") or not torch.cuda.is_available():
        return ["cpu"]
    if DEVICE != "cuda":
        return [DEVICE]
    if PARALLEL_GPUS:
        return [f"cuda:{i}" for i in range(torch.cuda.device_count())]
    return ["cuda:0"]


def variant_name(permutations):
    return "symmetrized" if bool(permutations) else "unconstrained"


def config_grid():
    return [
        {"task_mode": str(task_mode), "permutations": bool(permutations)}
        for task_mode in TASK_MODE
        for permutations in PERMUTATIONS
    ]


def run_config(task_mode, permutations, seed, device):
    return {
        "run": {
            "run_name": RUN_NAME,
            "out_root": str(Path(OUT_ROOT) / str(task_mode) / variant_name(permutations)),
            "run_id": None,
            "seed": int(seed),
            "device_requested": DEVICE,
            "parallel_gpus": PARALLEL_GPUS,
            "device_used": str(device),
            "test_run": TEST_RUN,
        },
        "task": {
            "mode": task_mode,
            "n_classes": N_CLASSES,
            "dims": DIMS,
            "points": POINTS,
            "semi_eta": SEMI_ETA,
            "semi_n_labeled": SEMI_N_LABELED,
            "semi_replace_unlabeled_with_noise": SEMI_REPLACE_UNLABELED_WITH_NOISE,
            "semi_train_point_choices": [int(v) for v in SEMI_TRAIN_POINT_CHOICES],
        },
        "model": {
            "n_embd": DIMS + N_CLASSES,
            "n_layer": N_LAYER,
            "n_head": N_HEAD,
            "act_attn": ACT_ATTN,
            "attn_mask": ATTN_MASK,
            "permutations": bool(permutations),
            "permutations_kfix": N_CLASSES,
        },
        "training": {
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "beta1": BETA1,
            "beta2": BETA2,
            "train_steps": min(TRAIN_STEPS, 100) if TEST_RUN else TRAIN_STEPS,
            "redraw_every_iters": REDRAW_EVERY_ITERS,
            "save_every_steps": SAVE_EVERY_STEPS,
        },
    }


def run_training(task_mode, permutations, run_idx=0, seed=None, device_override=None, progress_position=0):
    device = device_override or resolve_devices()[0]
    seed = int(SEED if seed is None else seed)
    if str(device).startswith("cuda"):
        torch.cuda.set_device(torch.device(device))
    set_seed(seed)

    run_id = f"{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
    out_dir = Path(OUT_ROOT) / str(task_mode) / variant_name(permutations) / run_id
    out_dir.mkdir(parents=True, exist_ok=True)

    cfg = run_config(task_mode, permutations, seed, device)
    cfg["run"]["run_id"] = run_id
    with open(out_dir / "config.yaml", "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    model = build_model(permutations=permutations).to(device)
    model.train()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        betas=(BETA1, BETA2),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    task = build_task(task_mode, BATCH_SIZE, POINTS, DIMS, device=device)

    train_steps = cfg["training"]["train_steps"]
    state_path = out_dir / "state.pt"
    loss_log = []
    z = y = None
    desc = f"{task_mode}/{variant_name(permutations)} run {run_idx} on {device}"
    pbar = tqdm(range(train_steps), desc=desc, position=progress_position, leave=True)

    for i in pbar:
        if i % REDRAW_EVERY_ITERS == 0:
            with torch.no_grad():
                task.resample()
                z, y = task.evaluate(n_points=sample_total_tokens(task_mode))

        loss = train_step(model, z, y, optimizer)
        pbar.set_postfix(loss=f"{loss:.6f}")

        if (i % SAVE_EVERY_STEPS == 0) or (i == train_steps - 1):
            loss_log.append(loss)
            torch.save({"model_state_dict": model.state_dict(), "loss_log": loss_log, "step": i}, state_path)

    print(f"Saved run to: {out_dir}")
    return out_dir, state_path, loss_log

## Run Grid

The final cell constructs the run grid, assigns jobs round-robin to the available devices, launches the jobs, and returns a record of the produced checkpoints. The output shown below is the run record from the checkpoint generation used for the paper.

In [4]:
def _run_one_record(job, device, progress_position):
    out_dir, state_path, loss_log = run_training(
        task_mode=job["task_mode"],
        permutations=job["permutations"],
        run_idx=job["run_idx"],
        seed=job["seed"],
        device_override=device,
        progress_position=progress_position,
    )
    return {
        "job_idx": int(job["job_idx"]),
        "config_idx": int(job["config_idx"]),
        "run_idx": int(job["run_idx"]),
        "seed": int(job["seed"]),
        "task_mode": str(job["task_mode"]),
        "permutations": bool(job["permutations"]),
        "variant": variant_name(job["permutations"]),
        "device": str(device),
        "out_dir": str(out_dir),
        "state_path": str(state_path),
        "n_logged_losses": len(loss_log),
        "last_logged_loss": (loss_log[-1] if loss_log else None),
    }


def _run_device_queue(device, jobs, progress_position):
    return [_run_one_record(job, device, progress_position) for job in jobs]


configs = config_grid()
jobs = []
for config_idx, cfg in enumerate(configs):
    for run_idx in range(int(N_RUNS)):
        jobs.append({
            "job_idx": len(jobs),
            "config_idx": config_idx,
            "run_idx": run_idx,
            "seed": int(SEED) + config_idx * int(N_RUNS) + run_idx,
            "task_mode": cfg["task_mode"],
            "permutations": cfg["permutations"],
        })

devices = resolve_devices()
device_jobs = {device: [] for device in devices}
for j, job in enumerate(jobs):
    device_jobs[devices[j % len(devices)]].append(job)

print("Configurations:")
for idx, cfg in enumerate(configs):
    print(f"{idx}: {cfg['task_mode']}/{variant_name(cfg['permutations'])}")
print(f"Using devices: {devices}")
for device, assigned in device_jobs.items():
    labels = [f"{j['task_mode']}/{variant_name(j['permutations'])}:run{j['run_idx']}" for j in assigned]
    print(f"{device}: {labels}")

all_runs = []
if len(devices) == 1:
    all_runs.extend(_run_device_queue(devices[0], device_jobs[devices[0]], progress_position=0))
else:
    with ThreadPoolExecutor(max_workers=len(devices)) as ex:
        futures = [
            ex.submit(_run_device_queue, device, device_jobs[device], pos)
            for pos, device in enumerate(devices)
        ]
        for fut in as_completed(futures):
            all_runs.extend(fut.result())

all_runs = sorted(all_runs, key=lambda r: r["job_idx"])
print("Saved run paths:")
for r in all_runs:
    print(
        f"[{r['task_mode']}/{r['variant']} run {r['run_idx']}] "
        f"seed={r['seed']} device={r['device']} path={r['out_dir']}"
    )

all_runs

Configurations:
0: voronoicell/symmetrized
1: voronoicell/unconstrained
2: semi_supervised/symmetrized
3: semi_supervised/unconstrained
Using devices: ['cuda:0', 'cuda:1', 'cuda:2', 'cuda:3']
cuda:0: ['voronoicell/symmetrized:run0', 'semi_supervised/symmetrized:run0']
cuda:1: ['voronoicell/symmetrized:run1', 'semi_supervised/symmetrized:run1']
cuda:2: ['voronoicell/unconstrained:run0', 'semi_supervised/unconstrained:run0']
cuda:3: ['voronoicell/unconstrained:run1', 'semi_supervised/unconstrained:run1']


voronoicell/symmetrized run 1 on cuda:1:   0%|          | 0/20001 [00:00<?, ?it/s]

voronoicell/unconstrained run 1 on cuda:3:   0%|          | 0/20001 [00:00<?, ?it/s]

voronoicell/unconstrained run 0 on cuda:2:   0%|          | 0/20001 [00:00<?, ?it/s]

voronoicell/symmetrized run 0 on cuda:0:   0%|          | 0/20001 [00:00<?, ?it/s]

...

semi_supervised/symmetrized run 1 on cuda:1:   0%|          | 0/20001 [00:00<?, ?it/s]

Saved run to: models/voronoicell/symmetrized/20260528_094905_0a9d799e


semi_supervised/symmetrized run 0 on cuda:0:   0%|          | 0/20001 [00:00<?, ?it/s]

Saved run to: models/semi_supervised/unconstrained/20260528_101057_9548fe47
Saved run to: models/semi_supervised/unconstrained/20260528_101058_ee0fa700
Saved run to: models/semi_supervised/symmetrized/20260528_102237_84b72bb2
Saved run to: models/semi_supervised/symmetrized/20260528_102245_9c7c8bd2
Saved run paths:
[voronoicell/symmetrized run 0] seed=4 device=cuda:0 path=models/voronoicell/symmetrized/20260528_094905_0a9d799e
[voronoicell/symmetrized run 1] seed=5 device=cuda:1 path=models/voronoicell/symmetrized/20260528_094905_47fbf5ff
[voronoicell/unconstrained run 0] seed=6 device=cuda:2 path=models/voronoicell/unconstrained/20260528_094905_de0b65de
[voronoicell/unconstrained run 1] seed=7 device=cuda:3 path=models/voronoicell/unconstrained/20260528_094905_beca9bb9
[semi_supervised/symmetrized run 0] seed=8 device=cuda:0 path=models/semi_supervised/symmetrized/20260528_102245_9c7c8bd2
[semi_supervised/symmetrized run 1] seed=9 device=cuda:1 path=models/semi_supervised/symmetrized/

[{'job_idx': 0,
  'config_idx': 0,
  'run_idx': 0,
  'seed': 4,
  'task_mode': 'voronoicell',
  'permutations': True,
  'variant': 'symmetrized',
  'device': 'cuda:0',
  'out_dir': 'models/voronoicell/symmetrized/20260528_094905_0a9d799e',
  'state_path': 'models/voronoicell/symmetrized/20260528_094905_0a9d799e/state.pt',
  'n_logged_losses': 21,
  'last_logged_loss': 0.3192550241947174},
 {'job_idx': 1,
  'config_idx': 0,
  'run_idx': 1,
  'seed': 5,
  'task_mode': 'voronoicell',
  'permutations': True,
  'variant': 'symmetrized',
  'device': 'cuda:1',
  'out_dir': 'models/voronoicell/symmetrized/20260528_094905_47fbf5ff',
  'state_path': 'models/voronoicell/symmetrized/20260528_094905_47fbf5ff/state.pt',
  'n_logged_losses': 21,
  'last_logged_loss': 0.30583369731903076},
 {'job_idx': 2,
  'config_idx': 1,
  'run_idx': 0,
  'seed': 6,
  'task_mode': 'voronoicell',
  'permutations': False,
  'variant': 'unconstrained',
  'device': 'cuda:2',
  'out_dir': 'models/voronoicell/unconstrain